# SafeVision Notebook 2: Worker matching + tests on ConstructionSite 10k

**Goal:** turn the detector's boxes into a **per-worker check** (helmet yes/no/unsure, vest yes/no/unsure), then test the whole vision part on **ConstructionSite 10k** [1], the benchmark where large ready-made vision-language models scored only **20.4% precision** on the PPE rule (human inspector: 95.6%).

| ID | Question | Data | Main scores |
|---|---|---|---|
| **E3a** | Does SafeVision (detector + box-based matching) catch helmet violations in whole images? | ConstructionSite 10k test, helmet cases | precision, recall, F1 |
| **E3b** | Does **pose-based** matching [4] do better than box-based matching? | same | same |
| **E3c** | Does it point at the right worker? | violator boxes from [1] | localisation hit rate |
| **E3d** | Does it recognise workers who **do** wear a helmet? | "worker with white hard hat" boxes from [1] | recall |
| **E4** | Where does it fail? (light, distance, view, image quality) | ConstructionSite 10k tags | recall and precision per condition |

**Needs:** the trained detector from Notebook 1 (`detector_best.pt`) and access to ConstructionSite 10k on Hugging Face (a gated dataset: needs an accepted access request and a read token).

**Fair tuning:** the two decision thresholds are tuned on a sample of the ConstructionSite **train** split. The **test** split is used only once, for the final numbers.

**Runtime:** GPU recommended (about 20-40 min for 3,004 test images plus the tuning sample).

## 0. Install libraries

`ultralytics` (YOLO11 detector + YOLO11-pose), `datasets` (to load ConstructionSite 10k from Hugging Face), `openpyxl` (Excel export).

In [ ]:
# [colab-only]
!nvidia-smi
%pip install -q "ultralytics>=8.3.0" datasets openpyxl huggingface_hub

## 1. Configuration (edit this cell only)

- `DETECTOR_WEIGHTS`: the best detector from Notebook 1, uploaded in Section 3.
- `IMGSZ`: use the same image size as the best Notebook 1 run (read automatically from `run_config.json` when available).
- `POSE_MODEL`: COCO-pretrained YOLO11-pose, used as-is (no training needed).
- `TUNE_N`: how many ConstructionSite **train** images to use for picking thresholds.
- `HELMET_WORDS`: words that mark a rule-1 violation as a **helmet** case. Rule 1 in [1] also covers shoes, clothing, vests (at night) and eye protection, so only helmet cases are kept.

In [ ]:
# CONFIG
from pathlib import Path
import re, json, random

OUT = Path("/content/safevision_outputs_nb2")
DETECTOR_WEIGHTS = Path("/content/detector_best.pt")
RUN_CONFIG = Path("/content/run_config.json")          # optional, from Notebook 1
IMGSZ = 960                                            # overwritten from run_config.json if present
POSE_MODEL = "yolo11s-pose.pt"
CS10K_HUB = "LouisChen15/ConstructionSite"
CS10K_LOCAL = None                                     # leave None on Colab (loads from the Hub)

TUNE_N = 400                                           # train images used to tune thresholds
TEST_N = None                                          # None = all test images
SEED = 42
DEVICE = 0
BATCH = 16

# Threshold grid searched on the tuning images (decision thresholds of the matching rules)
CONF_GRID = [0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

# A rule-1 reason that mentions any of these is a helmet case
HELMET_WORDS = re.compile(r"\b(hard ?hats?|hardhats?|helmets?|head protection|safety hats?)\b", re.I)

RES, FIGS, ERR = OUT / "results", OUT / "figures", OUT / "error_examples"
for p in [OUT, RES, FIGS, ERR]:
    p.mkdir(parents=True, exist_ok=True)
if RUN_CONFIG.exists():
    IMGSZ = json.load(open(RUN_CONFIG))["best_imgsz"]
print("Image size:", IMGSZ)

## 2. Results logger

Same format as Notebook 1, so all experiment tables can be merged into one Excel file at the end.

In [ ]:
# Results logger
import csv, datetime
import pandas as pd

LOG_CSV = RES / "experiments_log.csv"
LOG_FIELDS = ["experiment", "run", "model", "imgsz", "dataset", "split",
              "class_name", "metric", "value", "notes", "timestamp"]

def log_result(experiment, run, model, imgsz, dataset, split, class_name, metric, value, notes=""):
    # Append one measured number to the results log.
    new_file = not LOG_CSV.exists()
    with open(LOG_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=LOG_FIELDS)
        if new_file:
            w.writeheader()
        w.writerow(dict(experiment=experiment, run=run, model=model, imgsz=imgsz, dataset=dataset,
                        split=split, class_name=class_name, metric=metric, value=round(float(value), 4),
                        notes=notes, timestamp=datetime.datetime.now().isoformat(timespec="seconds")))

def read_log():
    return pd.read_csv(LOG_CSV) if LOG_CSV.exists() else pd.DataFrame(columns=LOG_FIELDS)

## 3. Get the detector and log in to Hugging Face

1. **Detector:** upload `detector_best.pt` (and optionally `run_config.json`) from the Notebook 1 ZIP. If Notebook 1 pushed its outputs to the private HF repo, they are downloaded from there instead.
2. **ConstructionSite 10k is gated:** access needs an accepted request on Hugging Face and a read token stored as the secret `HF_TOKEN`.

In [ ]:
# [colab-only]
from google.colab import files, userdata
from huggingface_hub import login, hf_hub_download, HfApi

token = userdata.get("HF_TOKEN")
login(token=token)

import shutil
user = HfApi(token=token).whoami()["name"]
repo = f"{user}/safevision-artifacts"
try:   # 1) final Notebook 1 outputs in the private artifacts repo
    for f in ["models/detector_best.pt", "run_config.json"]:
        p = hf_hub_download(repo, f"nb1_vision_detector/{f}", token=token)
        shutil.copy(p, Path("/content") / Path(f).name)
    print("Detector downloaded from the private HF repo (Notebook 1 outputs).")
except Exception as e:
    # 2) Notebook 1 not finished yet: use the saved training checkpoint (E1b at 960 px is the best model, then E1a)
    for run, size in [("E1b_yolo11s_960", 960), ("E1a_yolo11s_640", 640)]:
        try:
            p = hf_hub_download(repo, f"checkpoints/{run}/best_final.pt", token=token)
            shutil.copy(p, DETECTOR_WEIGHTS); IMGSZ = size
            print(f"Notebook 1 outputs not found; using the saved checkpoint {run} ({size} px).")
            break
        except Exception:
            continue
    else:   # 3) otherwise upload by hand
        print("HF download not possible (", e, "). Upload detector_best.pt (and run_config.json if available):")
        files.upload()

if RUN_CONFIG.exists():
    IMGSZ = json.load(open(RUN_CONFIG))["best_imgsz"]
assert DETECTOR_WEIGHTS.exists(), "detector_best.pt not found in /content"
print("Detector:", DETECTOR_WEIGHTS, "| image size:", IMGSZ)

## 4. Load ConstructionSite 10k

Loads the **test** split (3,004 images) and a random sample of `TUNE_N` **train** images. Each record holds the photo, a caption, the 4 rule-violation fields (each with violator boxes and a one-to-two sentence human reason), and 4 condition tags: illumination, camera distance, view and quality of information [1].

In [ ]:
# Load ConstructionSite 10k
from datasets import load_dataset, load_from_disk

def load_cs10k():
    # Load the dataset from a local copy (tests) or from the Hugging Face Hub.
    return load_from_disk(CS10K_LOCAL) if CS10K_LOCAL else load_dataset(CS10K_HUB)

cs = load_cs10k()
test_ds = cs["test"] if TEST_N is None else cs["test"].select(range(min(TEST_N, len(cs["test"]))))
rng = random.Random(SEED)
tune_idx = rng.sample(range(len(cs["train"])), k=min(TUNE_N, len(cs["train"])))
tune_ds = cs["train"].select(tune_idx)
print(cs)
print(f"Tuning images: {len(tune_ds)} | test images: {len(test_ds)}")

## 5. Build the ground truth for helmet violations

Each image gets one of three labels:
- **helmet_violation**: rule 1 is violated **and** the human reason mentions a hard hat or helmet (positive),
- **no_rule1_violation**: rule 1 is not violated (negative),
- **other_rule1**: rule 1 is violated for something else (shoes, clothing, eye protection...), so it is **excluded**, because our system only checks helmets for this rule.

We also convert the violator boxes to pixels. The format is auto-detected: values ≤ 1.5 are treated as relative coordinates. The drawings confirm this visually.

In [ ]:
# Build the ground truth for helmet violations
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def parse_violation(v):
    # Return (list_of_boxes, reason) from a rule-violation field (dict, JSON string or None).
    if v is None or v == "" or v == {}:
        return [], ""
    if isinstance(v, str):
        try:
            v = json.loads(v)
        except Exception:
            return [], v
    boxes = v.get("bounding_box") or []
    if boxes and isinstance(boxes[0], (int, float)):     # a single box given as 4 numbers
        boxes = [boxes]
    return [list(map(float, b)) for b in boxes if b and len(b) == 4], (v.get("reason") or "")

def to_pixels(b, W, H):
    # Convert an [x1, y1, x2, y2] box to pixels if it is given in relative (0-1) coordinates.
    return [b[0] * W, b[1] * H, b[2] * W, b[3] * H] if max(b) <= 1.5 else b

def gt_label(rec):
    # Label one record as helmet_violation / no_rule1_violation / other_rule1.
    boxes, reason = parse_violation(rec["rule_1_violation"])
    if not boxes and not reason:
        return "no_rule1_violation", [], ""
    return ("helmet_violation" if HELMET_WORDS.search(reason) else "other_rule1"), boxes, reason

def build_gt(ds):
    # Table with one row per image: id, label, violator boxes (pixels), reason and condition tags.
    rows = []
    for i in range(len(ds)):
        rec = ds[i]; W, H = rec["image"].size
        lab, boxes, reason = gt_label(rec)
        rows.append(dict(idx=i, image_id=rec["image_id"], label=lab, reason=reason, W=W, H=H,
                         boxes=[to_pixels(b, W, H) for b in boxes],
                         white_hat=[to_pixels(list(map(float, b)), W, H) for b in (rec.get("worker_with_white_hard_hat") or []) if b and len(b) == 4],
                         illumination=rec.get("illumination"), camera_distance=rec.get("camera_distance"),
                         view=rec.get("view"), quality_of_info=rec.get("quality_of_info")))
    return pd.DataFrame(rows)

gt_tune, gt_test = build_gt(tune_ds), build_gt(test_ds)
counts = pd.concat({"tune (train sample)": gt_tune.label.value_counts(), "test": gt_test.label.value_counts()}, axis=1).fillna(0).astype(int)
counts.to_csv(RES / "cs10k_label_counts.csv")
print(counts)

# Visual check: violator boxes on up to 3 positive test images.
pos = gt_test[gt_test.label == "helmet_violation"].head(3)
if len(pos):
    fig, axes = plt.subplots(1, len(pos), figsize=(6 * len(pos), 5), squeeze=False)
    for ax, (_, r) in zip(axes[0], pos.iterrows()):
        ax.imshow(test_ds[int(r.idx)]["image"]); ax.axis("off"); ax.set_title(r.reason[:70], fontsize=8)
        for b in r.boxes:
            ax.add_patch(patches.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, color="red", lw=2))
    plt.tight_layout(); plt.savefig(FIGS / "fig_cs10k_gt_check.png", dpi=110); plt.show()

## 6. Worker matching (step V2 of SafeVision)

This is the shared matching module (also saved as `matching.py` for the web app backend, so the app behaves exactly like the evaluated system).

- **Box method:** the head area is the top 30% of a person box, the torso area is 20-65%.
- **Pose method:** head and torso areas come from YOLO11-pose keypoints (nose, eyes, ears; shoulders, hips), following the pose-guided PPE assignment of López et al. [4]. If the keypoints are missing, it falls back to the box method.
- **Statuses:** `yes`, `no` or `unsure`. "Unsure" is shown in **yellow** in the app so a person checks it. This is our answer to the high false-alarm rate of ready-made models [1], [12].
- Bare heads with no detected body (crowds, blocked views) still count as workers without a helmet.

In [ ]:
"""SafeVision worker matching (step V2).

Turns raw detector boxes (person, head, helmet, vest) into one record per
worker with a helmet status and a vest status: "yes", "no" or "unsure".

Two ways to find where a worker's head and torso are:
  * "box":  fixed fractions of the person box (top 30% = head area,
            20-65% = torso area). Simple, no extra model.
  * "pose": body keypoints from YOLO11-pose (nose/eyes/ears = head,
            shoulders/hips = torso), following the idea of matching PPE
            to body parts with pose estimation (Lopez et al., 2025, paper [4]).
If pose keypoints are missing for a person, "pose" falls back to "box".

The same file is used by Notebook 2 (experiments E3/E4) and by the backend,
so the app behaves exactly like the evaluated system.
"""
from __future__ import annotations

CLASSES = ["person", "head", "helmet", "vest"]

# COCO keypoint indices used by YOLO11-pose
HEAD_KPTS = [0, 1, 2, 3, 4]          # nose, left/right eye, left/right ear
TORSO_KPTS = [5, 6, 11, 12]          # left/right shoulder, left/right hip


# ----------------------------------------------------------------- geometry
def area(b):
    """Area of an [x1, y1, x2, y2] box (0 if the box is empty)."""
    return max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])


def intersection(a, b):
    """Area where two boxes overlap."""
    return area([max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])])


def iou(a, b):
    """Intersection over union of two boxes (0 = no overlap, 1 = identical)."""
    inter = intersection(a, b)
    union = area(a) + area(b) - inter
    return inter / union if union > 0 else 0.0


def inside_fraction(inner, outer):
    """Share of `inner` that lies inside `outer` (e.g. how much of a helmet is in the head area)."""
    a = area(inner)
    return intersection(inner, outer) / a if a > 0 else 0.0


# ------------------------------------------------------------ detections
def detections_from_result(result, min_conf=0.10):
    """Convert one Ultralytics detection result into {class_name: [(box, conf), ...]}."""
    dets = {c: [] for c in CLASSES}
    names = result.names
    for box, conf, cls in zip(result.boxes.xyxy.tolist(), result.boxes.conf.tolist(), result.boxes.cls.tolist()):
        name = names[int(cls)]
        if name in dets and conf >= min_conf:
            dets[name].append((box, float(conf)))
    return dets


def poses_from_result(result, kpt_conf=0.5):
    """Convert one YOLO11-pose result into a list of (person_box, {kpt_index: (x, y)})."""
    out = []
    if result.keypoints is None:
        return out
    xy = result.keypoints.xy.tolist()
    kc = result.keypoints.conf.tolist() if result.keypoints.conf is not None else None
    for i, box in enumerate(result.boxes.xyxy.tolist()):
        pts = {}
        for k, (x, y) in enumerate(xy[i]):
            if kc is None or kc[i][k] >= kpt_conf:
                pts[k] = (x, y)
        out.append((box, pts))
    return out


# ------------------------------------------------------- body regions
def box_regions(p):
    """Head and torso areas as fixed fractions of a person box (method 'box')."""
    x1, y1, x2, y2 = p
    h = y2 - y1
    head = [x1, y1 - 0.05 * h, x2, y1 + 0.30 * h]      # a little above the box too: helmets stick out
    torso = [x1, y1 + 0.20 * h, x2, y1 + 0.65 * h]
    return head, torso


def pose_regions(p, pose_list):
    """Head and torso areas from the best-matching pose skeleton (method 'pose'); None if unavailable."""
    best, best_iou = None, 0.5                          # pose person must overlap the detected person well
    for pbox, pts in pose_list:
        v = iou(p, pbox)
        if v > best_iou:
            best, best_iou = pts, v
    if best is None:
        return None
    head_pts = [best[k] for k in HEAD_KPTS if k in best]
    torso_pts = [best[k] for k in TORSO_KPTS if k in best]
    if len(head_pts) < 2 or len(torso_pts) < 2:
        return None
    ph = p[3] - p[1]
    hx = [x for x, _ in head_pts]; hy = [y for _, y in head_pts]
    cx, cy = sum(hx) / len(hx), sum(hy) / len(hy)
    s = max(max(hx) - min(hx), max(hy) - min(hy), 0.08 * ph)
    head = [cx - 1.2 * s, cy - 2.0 * s, cx + 1.2 * s, cy + 0.8 * s]     # helmets sit above the eyes
    tx = [x for x, _ in torso_pts]; ty = [y for _, y in torso_pts]
    pad = 0.1 * (max(tx) - min(tx) + 1)
    torso = [min(tx) - pad, min(ty) - pad, max(tx) + pad, max(ty) + pad]
    return head, torso


def _best(items, region, min_inside, used):
    """Highest-confidence item (index, box, conf) mostly inside `region` and not used by another worker."""
    best = None
    for i, (b, c) in enumerate(items):
        if i in used:
            continue
        if inside_fraction(b, region) >= min_inside and (best is None or c > best[2]):
            best = (i, b, c)
    return best


# ------------------------------------------------------------ main function
def match_workers(dets, img_h, pose_list=None, method="box", conf_yes=0.40, conf_no=0.40,
                  min_person_conf=0.30):
    """Assign helmets, heads and vests to workers.

    Returns a list of worker dicts:
      box, source ('person' or 'head-only'), person_conf,
      helmet ('yes'/'no'/'unsure'), helmet_conf, vest ('yes'/'no'/'unsure'), vest_conf.

    Rules (helmet):  matched helmet with conf >= conf_yes            -> 'yes'
                     else matched bare head with conf >= conf_no      -> 'no'
                     else                                             -> 'unsure'
    Rules (vest):    matched vest with conf >= conf_yes               -> 'yes'
                     else person clearly visible (conf >= 0.5 and
                          at least 15% of the image height)           -> 'no'
                     else                                             -> 'unsure'
    Bare heads that belong to no detected person (occlusion, crowds) become
    extra 'head-only' workers, so a hidden body does not hide a missing helmet.
    """
    workers = []
    used = {"head": set(), "helmet": set(), "vest": set()}
    persons = sorted([p for p in dets["person"] if p[1] >= min_person_conf], key=lambda t: -t[1])
    for pbox, pconf in persons:
        regions = pose_regions(pbox, pose_list) if (method == "pose" and pose_list) else None
        head_r, torso_r = regions if regions else box_regions(pbox)
        helmet = _best(dets["helmet"], head_r, 0.5, used["helmet"])
        head = _best(dets["head"], head_r, 0.5, used["head"])
        vest = _best(dets["vest"], torso_r, 0.3, used["vest"])
        for key, hit in (("helmet", helmet), ("head", head), ("vest", vest)):
            if hit:
                used[key].add(hit[0])
        if helmet and helmet[2] >= conf_yes:
            h_status, h_conf = "yes", helmet[2]
        elif head and head[2] >= conf_no:
            h_status, h_conf = "no", head[2]
        else:
            h_status, h_conf = "unsure", max([x[2] for x in (helmet, head) if x] or [0.0])
        visible = pconf >= 0.5 and (pbox[3] - pbox[1]) >= 0.15 * img_h
        if vest and vest[2] >= conf_yes:
            v_status, v_conf = "yes", vest[2]
        elif visible:
            v_status, v_conf = "no", pconf
        else:
            v_status, v_conf = "unsure", vest[2] if vest else 0.0
        workers.append(dict(box=pbox, source="person", person_conf=pconf, helmet=h_status,
                            helmet_conf=h_conf, vest=v_status, vest_conf=v_conf))
    # Bare heads with no person around them -> extra workers without a helmet.
    for i, (hb, hc) in enumerate(dets["head"]):
        if i in used["head"] or hc < conf_no:
            continue
        if any(iou(hb, hl) > 0.3 or inside_fraction(hl, hb) > 0.5 for hl, _ in dets["helmet"]):
            continue                                       # a helmet sits on this head
        workers.append(dict(box=hb, source="head-only", person_conf=0.0, helmet="no",
                            helmet_conf=hc, vest="unsure", vest_conf=0.0))
    return workers


def image_has_helmet_violation(workers):
    """Image-level decision: at least one worker clearly without a helmet."""
    return any(w["helmet"] == "no" for w in workers)

## 7. Run the detector and the pose model once, and cache the raw results

Each image goes through the detector (and the pose model) **once**, with a low confidence floor (0.10). Every threshold and method in the experiments below then reuses these cached boxes, so the experiments are fast and exactly comparable.

In [ ]:
# Run the detector and the pose model once, and cache the raw results
import torch
from ultralytics import YOLO

if not torch.cuda.is_available():
    DEVICE = "cpu"

detector = YOLO(str(DETECTOR_WEIGHTS))
pose_model = YOLO(POSE_MODEL)

def run_models(ds, tag):
    # Return a list (one entry per image) of {'dets': ..., 'poses': ..., 'H': ...}.
    cache = []
    for start in range(0, len(ds), BATCH):
        imgs = [ds[i]["image"].convert("RGB") for i in range(start, min(start + BATCH, len(ds)))]
        det_res = detector.predict(imgs, imgsz=IMGSZ, conf=0.10, device=DEVICE, verbose=False)
        pose_res = pose_model.predict(imgs, imgsz=IMGSZ, conf=0.25, device=DEVICE, verbose=False)
        for im, d, p in zip(imgs, det_res, pose_res):
            cache.append({"dets": detections_from_result(d), "poses": poses_from_result(p), "H": im.size[1]})
        print(f"\r{tag}: {len(cache)}/{len(ds)} images", end="")
    print()
    return cache

cache_tune = run_models(tune_ds, "tune")
cache_test = run_models(test_ds, "test")

## 8. Scoring helpers

For every image we predict "helmet violation: yes/no" (any worker with helmet status `no`) and compare with the ground truth from Section 5. Images labelled *other_rule1* are left out.

- **Precision:** of the images we flagged, how many really had a helmet violation?
- **Recall:** of the images with a helmet violation, how many did we flag?
- **F1:** the balance of the two.

In [ ]:
# Scoring helpers
def predict_images(cache, method, conf_yes, conf_no):
    # Run worker matching on every cached image; return (list of worker lists, list of 0/1 flags).
    workers = [match_workers(c["dets"], c["H"], c["poses"], method=method, conf_yes=conf_yes, conf_no=conf_no) for c in cache]
    return workers, [int(image_has_helmet_violation(w)) for w in workers]

def score(gt, flags, mask=None):
    # Precision / recall / F1 / accuracy over images with a clear label (optionally a subset mask).
    df = gt.assign(pred=flags)
    df = df[df.label != "other_rule1"]
    if mask is not None:
        df = df[mask.loc[df.index]]
    y = (df.label == "helmet_violation").astype(int)
    tp = int(((df.pred == 1) & (y == 1)).sum()); fp = int(((df.pred == 1) & (y == 0)).sum())
    fn = int(((df.pred == 0) & (y == 1)).sum()); tn = int(((df.pred == 0) & (y == 0)).sum())
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    acc = (tp + tn) / max(1, len(df))
    return dict(precision=p, recall=r, F1=f1, accuracy=acc, TP=tp, FP=fp, FN=fn, TN=tn, n=len(df))

## 9. Tune the two thresholds on the train sample (both methods)

Grid search over `conf_yes` (how sure the model must be that a helmet is worn) and `conf_no` (how sure it must be that a head is bare). The pair with the best **F1 on the tuning images** is kept for each method. The heat maps go in the report's appendix.

In [ ]:
# Tune the two thresholds on the train sample (both methods)
import numpy as np

best_params, tune_tables = {}, {}
for method in ["box", "pose"]:
    grid = np.zeros((len(CONF_GRID), len(CONF_GRID)))
    best = (-1, None)
    for i, cy in enumerate(CONF_GRID):
        for j, cn in enumerate(CONF_GRID):
            _, flags = predict_images(cache_tune, method, cy, cn)
            s = score(gt_tune, flags)
            grid[i, j] = s["F1"]
            if s["F1"] > best[0]:
                best = (s["F1"], (cy, cn))
    best_params[method] = best[1]
    tune_tables[method] = pd.DataFrame(grid, index=[f"yes>={c}" for c in CONF_GRID], columns=[f"no>={c}" for c in CONF_GRID])
    print(f"{method}: best tuning F1 = {best[0]:.3f} at conf_yes={best[1][0]}, conf_no={best[1][1]}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, m in zip(axes, ["box", "pose"]):
    im = ax.imshow(tune_tables[m].values, cmap="viridis", vmin=0, vmax=max(1e-6, tune_tables[m].values.max()))
    ax.set_xticks(range(len(CONF_GRID)), CONF_GRID); ax.set_yticks(range(len(CONF_GRID)), CONF_GRID)
    ax.set_xlabel("conf_no (bare head)"); ax.set_ylabel("conf_yes (helmet)"); ax.set_title(f"Tuning F1, {m} matching")
    plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.savefig(FIGS / "fig_E3_threshold_tuning.png", dpi=150); plt.show()
json.dump(best_params, open(RES / "matching_thresholds.json", "w"), indent=2)

## 10. Experiments E3a + E3b: Final test scores, compared with published results

The tuned settings are applied **once** to the test split. The reference rows are copied from the papers. They are **not** the same test set (paper [1] scores all PPE items on the whole test split; paper [8] used 100 images), so present them as context, not a head-to-head race.

In [ ]:
# Experiments E3a + E3b: Final test scores, compared with published results
test_workers, test_flags = {}, {}
rows = []
for method, exp in [("box", "E3a"), ("pose", "E3b")]:
    cy, cn = best_params[method]
    test_workers[method], test_flags[method] = predict_images(cache_test, method, cy, cn)
    s = score(gt_test, test_flags[method])
    for k in ["precision", "recall", "F1", "accuracy", "TP", "FP", "FN", "TN", "n"]:
        log_result("E3", f"{exp}_{method}_matching", "detector+" + method, IMGSZ, "ConstructionSite10k", "test",
                   "helmet", k, s[k], notes=f"conf_yes={cy}, conf_no={cn}; other_rule1 excluded")
    rows.append({"system": f"SafeVision ({method} matching)", **{k: round(s[k], 3) for k in ['precision', 'recall', 'F1']}, "test set": "CS10k test, helmet cases"})

refs = [("GPT-4V, zero-shot [1]", 0.204, 0.764, "CS10k test, all rule-1 PPE"),
        ("Human inspector [1]", 0.956, 0.666, "CS10k test, all rule-1 PPE"),
        ("Qwen2-VL-2B, single prompt [8]", 0.660, 0.673, "100 CS10k images"),
        ("Molmo-7B, single prompt [8]", 0.857, 0.480, "100 CS10k images")]
for name, p, r, note in refs:
    rows.append({"system": name, "precision": p, "recall": r, "F1": round(2 * p * r / (p + r), 3), "test set": note})
compare = pd.DataFrame(rows)
compare.to_csv(RES / "E3_comparison_with_papers.csv", index=False)
display(compare) if "display" in globals() else print(compare.to_string())

## 11. Experiment E3c: Does it point at the right worker?

For every human-marked violator box in the positive test images, we check whether SafeVision marked a worker **without a helmet** in the same place. Two measures:
- **IoU ≥ 0.5** with the worker's box (strict, standard detection overlap),
- **centre hit:** the centre of the human box lies inside our worker box (lenient; human boxes in [1] sometimes cover only part of the person).

In [ ]:
# Experiment E3c: Does it point at the right worker?
def localisation(gt, workers):
    # Share of human violator boxes matched by a 'no helmet' worker box (IoU>=0.5 and centre-hit).
    hits_iou = hits_ctr = total = 0
    for _, r in gt[gt.label == "helmet_violation"].iterrows():
        mine = [w["box"] for w in workers[int(r.idx)] if w["helmet"] == "no"]
        for b in r.boxes:
            total += 1
            cx, cy = (b[0] + b[2]) / 2, (b[1] + b[3]) / 2
            hits_iou += any(iou(b, m) >= 0.5 for m in mine)
            hits_ctr += any(m[0] <= cx <= m[2] and m[1] <= cy <= m[3] for m in mine)
    return dict(iou50_hit_rate=hits_iou / max(1, total), centre_hit_rate=hits_ctr / max(1, total), n_boxes=total)

for method in ["box", "pose"]:
    loc = localisation(gt_test, test_workers[method])
    for k, v in loc.items():
        log_result("E3", f"E3c_localisation_{method}", "detector+" + method, IMGSZ, "ConstructionSite10k", "test", "helmet", k, v)
    print(method, {k: round(v, 3) for k, v in loc.items()})

## 12. Experiment E3d: Does it recognise workers who wear a helmet?

ConstructionSite 10k marks every **worker with a white hard hat** [1]. A good system should give these workers helmet status `yes`. Recall = share of these boxes overlapped (centre-hit) by one of our `yes` workers. This checks the other side of the coin: not crying wolf on compliant workers.

In [ ]:
# Experiment E3d: Does it recognise workers who wear a helmet?
def white_hat_recall(gt, workers):
    # Share of 'worker with white hard hat' boxes covered by a worker we marked helmet = yes.
    hit = total = 0
    for _, r in gt.iterrows():
        mine = [w["box"] for w in workers[int(r.idx)] if w["helmet"] == "yes"]
        for b in r.white_hat:
            total += 1
            cx, cy = (b[0] + b[2]) / 2, (b[1] + b[3]) / 2
            hit += any(m[0] <= cx <= m[2] and m[1] <= cy <= m[3] for m in mine)
    return hit / max(1, total), total

for method in ["box", "pose"]:
    rec, n = white_hat_recall(gt_test, test_workers[method])
    log_result("E3", f"E3d_white_hat_{method}", "detector+" + method, IMGSZ, "ConstructionSite10k", "test", "helmet", "recall", rec, notes=f"{n} boxes")
    print(f"{method}: white-hard-hat recall = {rec:.3f} over {n} boxes")

## 13. Experiment E4: Robustness: where does it fail?

Splits the E3 test result by each condition tag of ConstructionSite 10k: **illumination, camera distance, view, quality of information** [1]. Recent work shows that safety AI can change a lot when conditions change [15], so we report where SafeVision is weaker instead of hiding it. Groups with fewer than 10 usable images are marked as too small.

In [ ]:
# Experiment E4: Robustness: where does it fail?
best_method = max(["box", "pose"], key=lambda m: score(gt_test, test_flags[m])["F1"])
print("Best method on test (for E4):", best_method)
e4_rows = []
for tag in ["illumination", "camera_distance", "view", "quality_of_info"]:
    for val in sorted(gt_test[tag].dropna().unique()):
        s = score(gt_test, test_flags[best_method], mask=(gt_test[tag] == val))
        e4_rows.append({"condition": tag, "value": val, **{k: s[k] for k in ["precision", "recall", "F1", "n"]}})
        if s["n"] >= 10:
            for k in ["precision", "recall", "F1", "n"]:
                log_result("E4", f"E4_{best_method}", "detector+" + best_method, IMGSZ, "ConstructionSite10k", "test",
                           f"{tag}={val}", k, s[k])
e4 = pd.DataFrame(e4_rows)
e4["note"] = np.where(e4.n < 10, "too few images", "")
e4.to_csv(RES / "E4_robustness.csv", index=False)
display(e4) if "display" in globals() else print(e4.to_string())

plot = e4[e4.n >= 10]
if len(plot):
    fig, ax = plt.subplots(figsize=(11, 4))
    labels = plot.condition + "\n" + plot.value.astype(str)
    x = np.arange(len(plot))
    ax.bar(x - 0.2, plot.recall, 0.4, label="recall"); ax.bar(x + 0.2, plot.precision, 0.4, label="precision")
    ax.set_xticks(x, labels, rotation=45, ha="right", fontsize=8); ax.set_ylim(0, 1); ax.legend()
    ax.set_title(f"E4: helmet-violation detection by condition ({best_method} matching)")
    plt.tight_layout(); plt.savefig(FIGS / "fig_E4_robustness.png", dpi=150); plt.show()

## 14. Example figures and error examples

- `fig_worker_matching_examples.png`: workers drawn in **green** (helmet yes), **red** (no) and **yellow** (unsure), the same colours as the web app.
- `error_examples/`: up to 10 missed violations (false negatives) and 10 false alarms (false positives) with our boxes drawn and the human reason in the title. Baranika uses these for the error-analysis section.

In [ ]:
# Example figures and error examples
STATUS_COLOR = {"yes": "lime", "no": "red", "unsure": "yellow"}

def draw_workers(ax, img, workers, title=""):
    # Draw each worker box in the colour of its helmet status.
    ax.imshow(img); ax.axis("off"); ax.set_title(title, fontsize=8)
    for w in workers:
        b = w["box"]
        ax.add_patch(patches.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, lw=2, color=STATUS_COLOR[w["helmet"]]))
        ax.text(b[0], b[1] - 3, f"helmet:{w['helmet']} vest:{w['vest']}", color=STATUS_COLOR[w["helmet"]], fontsize=6,
                bbox=dict(facecolor="black", alpha=0.5, pad=1))

W_ = test_workers[best_method]
df = gt_test.assign(pred=test_flags[best_method])
pick = list(df[(df.label == "helmet_violation") & (df.pred == 1)].idx[:3]) + list(df[(df.label == "no_rule1_violation") & (df.pred == 0)].idx[:3])
if pick:
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    for ax, i in zip(axes.ravel(), pick):
        draw_workers(ax, test_ds[int(i)]["image"], W_[int(i)], gt_test.loc[gt_test.idx == i, "label"].values[0])
    for ax in axes.ravel()[len(pick):]:
        ax.axis("off")
    plt.tight_layout(); plt.savefig(FIGS / "fig_worker_matching_examples.png", dpi=120); plt.show()

for kind, sel in [("false_negative", (df.label == "helmet_violation") & (df.pred == 0)),
                  ("false_positive", (df.label == "no_rule1_violation") & (df.pred == 1))]:
    for i in list(df[sel].idx[:10]):
        fig, ax = plt.subplots(figsize=(7, 5))
        r = gt_test[gt_test.idx == i].iloc[0]
        draw_workers(ax, test_ds[int(i)]["image"], W_[int(i)], f"{kind} | {r.image_id} | {r.reason[:80]}")
        plt.tight_layout(); plt.savefig(ERR / f"{kind}_{r.image_id}.png", dpi=100); plt.close(fig)
print("Error examples saved to", ERR)

## 15. Save the results (Excel + ZIP)

`E3_E4_worker_matching.xlsx` holds one sheet per experiment, the comparison with published results, the robustness table and the label counts.

In [ ]:
# Save the results (Excel + ZIP)
import shutil
log = read_log()
xlsx = RES / "E3_E4_worker_matching.xlsx"
with pd.ExcelWriter(xlsx) as xw:
    for exp in sorted(log.experiment.unique()):
        log[log.experiment == exp].pivot_table(index=["run", "model", "dataset", "split", "class_name"],
                                               columns="metric", values="value").reset_index().to_excel(xw, sheet_name=exp, index=False)
    compare.to_excel(xw, sheet_name="E3_vs_papers", index=False)
    e4.to_excel(xw, sheet_name="E4_table", index=False)
    counts.to_excel(xw, sheet_name="label_counts")
    log.to_excel(xw, sheet_name="full_log", index=False)
json.dump({"imgsz": IMGSZ, "pose_model": POSE_MODEL, "tune_n": len(tune_ds), "test_n": len(test_ds),
           "thresholds": best_params, "best_method": best_method, "seed": SEED},
          open(OUT / "run_config_nb2.json", "w"), indent=2)
archive = shutil.make_archive(str(OUT.parent / "safevision_nb2_outputs"), "zip", OUT)
print("Saved", xlsx, "| ZIP:", archive)

In [ ]:
# [colab-only]
from google.colab import files
files.download(archive)
try:
    api = HfApi(token=token)
    api.upload_folder(folder_path=str(OUT), repo_id=f"{api.whoami()['name']}/safevision-artifacts", path_in_repo="nb2_worker_matching")
    print("Uploaded to the private HF repo.")
except Exception as e:
    print("HF upload skipped:", e)

## References

Papers used directly in this notebook are **in bold**.

1. **Chen, X., & Zou, Z. (2026). Are large pre-trained vision language models effective construction safety inspectors? *Data-Centric Engineering*. https://doi.org/10.1017/dce.2026.10044 (arXiv:2508.11011)** (ConstructionSite 10k dataset, rules, GPT-4V and human baselines)
4. **López, L., Suárez-Ramírez, J., Alemán-Flores, M., & Monzón, N. (2025). Automated PPE compliance monitoring in industrial environments using deep learning-based detection and pose estimation. *Automation in Construction*, 176, 106231. https://doi.org/10.1016/j.autcon.2025.106231** (pose-guided PPE-to-worker matching)
8. **Sahraoui, I. (2025). *Automated hazard detection in construction sites using large language and vision-language models* [Master's thesis, University of Houston]. arXiv:2511.15720.** (small open VLM baselines on ConstructionSite 10k)
12. Adil, M., Ahmed, M., Aqib, M., Gonzalez, V. A., Lee, G., & Mei, Q. (2026). Integration of object detection and small VLMs for construction safety hazard identification. arXiv:2604.05210. (detector guidance cuts false alarms)
15. **Cui, Y., Wang, Z., Xu, Y., Cai, Q., Yao, H., Jiang, S., Zhong, B., & Xiong, H. (2026). SafeBuild-Bench: A temporal-robust construction safety benchmark with graph-enhanced data mining. KDD 2026. arXiv:2608.00068.** (robustness across conditions)
- Ultralytics YOLO11 and YOLO11-pose: https://docs.ultralytics.com; Hugging Face Datasets: https://huggingface.co/docs/datasets